# Process M1 DMS data from Hom et al.

Import Python modules

In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from Bio import SeqIO
import subprocess

In [2]:
import yaml

_config_dir = os.path.dirname(os.path.abspath("../config.yaml"))
with open("../config.yaml") as _f:
    _config = yaml.safe_load(_f)
data_dir = os.path.normpath(os.path.join(_config_dir, _config["data_dir"]))

The M1 CDS occupies nt 1–759 of the MP segment reference. Extract the M1 protein from that region and align it to the DMS protein (from PR8-M1.fasta) to build a site coordinate mapping. DMS effects are computed as log(preference / wt_preference).

In [3]:
# Read in the M1 reference protein from the MP segment reference
# M1 CDS = nt 1-759 (0-indexed: 0:759)
ref_fasta = os.path.join(data_dir, 'MP/all/curated_reference.fasta')
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq[:759].translate()).replace('*', '')
print('Reference M1 sequence length:', len(ref_aa_seq))

# Read in the DMS protein sequence from the PR8-M1 FASTA
dms_fasta = '../data/dms_data/Hom_M1/PR8-M1.fasta'
dms_seq = SeqIO.read(dms_fasta, 'fasta').seq
dms_aa_seq = str(dms_seq.translate()).replace('*', '')
print('DMS M1 sequence length:', len(dms_aa_seq))

# Load preferences; AA columns are all columns except 'site'
prefs = pd.read_csv('../data/dms_data/Hom_M1/summary_avgprefs.csv')
assert len(prefs) == len(dms_aa_seq), "Number of sites in preferences does not match DMS sequence length"
aa_cols = [c for c in prefs.columns if c != 'site']
print('DMS site range:', prefs['site'].min(), '-', prefs['site'].max())

# Add wildtype AA from the DMS protein sequence (site is 1-indexed)
prefs['wt_aa'] = prefs['site'].apply(lambda s: dms_aa_seq[s - 1])

# Melt to long format
m1_dms_data = prefs.melt(
    id_vars=['site', 'wt_aa'], value_vars=aa_cols,
    var_name='mut_aa', value_name='preference'
)

# Extract wt preferences and merge back
wt_data = m1_dms_data[m1_dms_data['wt_aa'] == m1_dms_data['mut_aa']][['site', 'preference']].copy()
wt_data.rename(columns={'preference': 'wt_preference'}, inplace=True)
m1_dms_data = m1_dms_data.merge(wt_data, on='site')

# Compute DMS effect as log(preference / wt_preference)
m1_dms_data['dms_effect'] = np.log(m1_dms_data['preference'] / m1_dms_data['wt_preference'])

# Rename site to dms_site
m1_dms_data = m1_dms_data.rename(columns={'site': 'dms_site'})

# Save sequences to a FASTA file for alignment
output_dir = '../results/dms_data/Hom_M1/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n')
        f.write(ref_aa_seq + '\n')
        f.write('>dms\n')
        f.write(dms_aa_seq + '\n')

# Align the sequences using MUSCLE
aligned_fasta = os.path.join(output_dir, 'aligned.fasta')
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

Reference M1 sequence length: 252
DMS M1 sequence length: 252
DMS site range: 1 - 252


In [4]:
# Read in the aligned sequences
seqs_dict = {}
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

# Compute percent identity
n_sites_to_compare = 0
n_identical = 0
for (ref_aa, dms_aa) in zip(aligned_ref_seq, aligned_dms_seq):
    if ref_aa != '-' and dms_aa != '-':
        n_sites_to_compare += 1
        if ref_aa == dms_aa:
            n_identical += 1

print(n_sites_to_compare, n_identical, n_identical / n_sites_to_compare)

# Determine numbering scheme
dms_start_index = int(prefs['site'].min())
numbering_dict = defaultdict(list)
assert len(aligned_ref_seq) == len(aligned_dms_seq)
for (seq_id, seq, start_index) in [
    ('tree_reference_site', aligned_ref_seq, 1),
    ('dms_site', aligned_dms_seq, dms_start_index),
]:
    seq_index = start_index
    for (alignment_index, aa) in enumerate(seq, 1):
        if aa != '-':
            numbering_dict['alignment_index'].append(alignment_index)
            numbering_dict['seq_id'].append(seq_id)
            numbering_dict['seq_index'].append(seq_index)
            numbering_dict['seq_aa'].append(aa)
            seq_index += 1

alignment_numbering_df = (
    pd.DataFrame(numbering_dict)
    .pivot(index='alignment_index', columns='seq_id', values='seq_index')
    .reset_index()
    .rename_axis(None, axis=1)
    .dropna()
    [['dms_site', 'tree_reference_site']]
)
alignment_numbering_df['dms_site'] = alignment_numbering_df['dms_site'].astype(int)
alignment_numbering_df['tree_reference_site'] = alignment_numbering_df['tree_reference_site'].astype(int)
alignment_numbering_df.head()

252 245 0.9722222222222222


,dms_site,tree_reference_site
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5


In [5]:
# Merge DMS data with numbering scheme and save
m1_dms_data_processed = (
    m1_dms_data
    .merge(alignment_numbering_df, on='dms_site', validate='many_to_one')
)

m1_dms_data_processed = m1_dms_data_processed[['dms_site', 'wt_aa', 'mut_aa', 'tree_reference_site', 'dms_effect']]
m1_dms_data_processed.to_csv(os.path.join(output_dir, 'processed_dms_data.csv'), index=False)
print('Number of mutations with processed data:', len(m1_dms_data_processed))
m1_dms_data_processed.head()

Number of mutations with processed data: 5040


,dms_site,wt_aa,mut_aa,tree_reference_site,dms_effect
0,1,M,A,1,-4.468899
1,2,S,A,2,-2.969077
2,3,L,A,3,-2.848604
3,4,L,A,4,-3.108507
4,5,T,A,5,-3.671066


In [6]:
m1_dms_data_processed.tail()

,dms_site,wt_aa,mut_aa,tree_reference_site,dms_effect
5035,248,M,Y,248,-1.828892
5036,249,Q,Y,249,-1.413800
5037,250,R,Y,250,-1.730334
5038,251,F,Y,251,-1.510559
5039,252,K,Y,252,-2.196348


In [7]:
sum(m1_dms_data_processed['dms_site'] != m1_dms_data_processed['tree_reference_site'])

0